# Notebook 4: SHAP Feature-Level Analysis

**Purpose**: Compute feature-level SHAP values to understand which CNN features drive predictions.

This notebook:
1. Initializes SHAP TreeExplainer for the trained XGBoost model
2. Computes SHAP values for test set
3. Generates feature importance visualizations
4. Creates individual prediction explanations

In [ ]:
import numpy as np
import shap
import xgboost as xgb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt

print(f"SHAP version: {shap.__version__}")

## Step 1: Load Trained Model and Test Features

In [ ]:
# Load model
model_path = Path("../models/xgb_model.pkl")
with open(model_path, 'rb') as f:
    xgb_model = pickle.load(f)

print("Model loaded")

# Load test features
features_dir = Path("../embeddings")
test_data = np.load(features_dir / "test_features.npz", allow_pickle=True)
X_test = test_data['features']
y_test = test_data['labels']

print(f"Test features shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

## Step 2: Initialize SHAP TreeExplainer

In [ ]:
# TreeExplainer is fast and exact for tree-based models
print("Initializing SHAP TreeExplainer...")
explainer = shap.TreeExplainer(xgb_model)

print(f"Expected prediction (base value): {explainer.expected_value}")

## Step 3: Compute SHAP Values for Test Set

**Note**: This may take a few minutes for large datasets.

In [ ]:
# Compute SHAP values
print("Computing SHAP values for test set...")
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Sample SHAP values (first instance): {shap_values[0, :5]}")

## Step 4: Summary Plot - Bar Chart (Overall Feature Importance)

In [ ]:
# Bar plot of mean absolute SHAP values
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("Feature Importance (Mean |SHAP| values)")
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.show()

print("Feature importance bar plot generated")

## Step 5: Summary Plot - Detailed View (Beeswarm)

In [ ]:
# Detailed SHAP summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, show=False, max_display=15)
plt.title("SHAP Summary Plot - Feature Impact on Predictions")
plt.tight_layout()
plt.show()

print("Detailed SHAP summary plot generated")

## Step 6: Top Important Features Analysis

In [ ]:
# Identify top important features by mean absolute SHAP value
feature_importance = np.abs(shap_values).mean(axis=0)
top_k = 10
top_indices = np.argsort(feature_importance)[-top_k:][::-1]

print(f"Top {top_k} most important features (by SHAP):")
for rank, idx in enumerate(top_indices, 1):
    print(f"{rank:2d}. Feature {idx}: {feature_importance[idx]:.4f}")

## Step 7: Dependence Plots for Top Features

In [ ]:
# Create dependence plots for top 3 features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, idx in enumerate(top_indices[:3]):
    shap.dependence_plot(
        idx,
        shap_values,
        X_test,
        ax=axes[i],
        show=False,
        title=f"Feature {idx} Dependence Plot"
    )

plt.tight_layout()
plt.show()

print("Dependence plots generated")

## Step 8: Individual Prediction Explanations (Force Plots)

In [ ]:
# Explain a few individual predictions
print("Force plots for individual predictions:")

# Select diverse samples
indices_to_explain = [0, 50, 100]  # Adjust based on dataset size

for idx in indices_to_explain:
    if idx < len(X_test):
        print(f"\n--- Sample {idx} ---")
        print(f"True label: {y_test[idx]} ({'NORMAL' if y_test[idx] == 0 else 'PNEUMONIA'})")
        
        prediction = xgb_model.predict([X_test[idx]])[0]
        prediction_proba = xgb_model.predict_proba([X_test[idx]])[0][1]
        print(f"Predicted label: {prediction} ({'NORMAL' if prediction == 0 else 'PNEUMONIA'})")
        print(f"Prediction probability: {prediction_proba:.4f}")
        
        # Force plot (may not render in all notebook environments)
        # shap.force_plot(explainer.expected_value, shap_values[idx], X_test[idx])
        
        # Alternative: waterfall plot
        plt.figure(figsize=(10, 5))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[idx],
                base_values=explainer.expected_value,
                data=X_test[idx]
            ),
            show=False,
            max_display=15
        )
        plt.title(f"Sample {idx} - Prediction Explanation")
        plt.tight_layout()
        plt.show()

## Step 9: Save SHAP Values for Data Valuation

In [ ]:
# Save SHAP values for use in data valuation
output_dir = Path("../shap_values")
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / "shap_values_test.npy", shap_values)
np.save(output_dir / "expected_value.npy", np.array([explainer.expected_value]))

print(f"SHAP values saved to {output_dir}")

## Summary

Feature-level SHAP analysis complete. Key findings:
- Most important CNN features identified
- Feature interactions visualized
- Individual predictions explained

Ready for data-level Shapley value computation in Notebook 5.